# Structured streaming Demo

In [1]:
# Imports cell

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}") \
    .getOrCreate()


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [6]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [7]:
from pyspark.sql import functions as F

events_df = parsed_df \
    .withColumn("event_time", F.timestamp_seconds("timestamp")) \
    .withColumn("size_delta", F.col("length.new") - F.coalesce(F.col("length.old"), F.lit(0)))

In [8]:
tumbling_df = events_df \
    .groupBy(F.window("event_time", "10 minutes"), "wiki") \
    .agg(
        F.count("*").alias("edits"),
        F.sum(F.col("bot").cast("int")).alias("bot_edits"),
        F.sum("size_delta").alias("net_bytes")
    )

tumbling_query = tumbling_df \
    .writeStream \
    .format("memory") \
    .queryName("tumbling_10min") \
    .outputMode("complete") \
    .start()

tumbling_query.processAllAvailable()

In [9]:
tumbling_table = spark.table("tumbling_10min") \
    .groupBy(F.col("window.start").alias("window_start"), F.col("window.end").alias("window_end")) \
    .pivot("wiki") \
    .sum("edits") \
    .fillna(0)

tumbling_table \
    .withColumn("total", sum(F.col(c) for c in tumbling_table.columns if c.endswith("wiki"))) \
    .orderBy("window_start") \
    .toPandas()

,window_start,window_end,dewiki,enwiki,eswiki,frwiki,total
0,2026-10-08 12:10:00,2026-10-08 12:20:00,116,502,39,116,773
1,2026-10-08 12:20:00,2026-10-08 12:30:00,110,518,64,122,814
2,2026-10-08 12:30:00,2026-10-08 12:40:00,99,640,60,121,920
3,2026-10-08 12:40:00,2026-10-08 12:50:00,118,663,64,133,978
4,2026-10-08 12:50:00,2026-10-08 13:00:00,12,82,5,18,117


In [10]:
sliding_df = events_df \
    .groupBy(F.window("event_time", "30 minutes", "10 minutes")) \
    .agg(
        F.count("*").alias("edits"),
        F.approx_count_distinct("user").alias("users"),
        F.approx_count_distinct("title").alias("pages"),
        F.round(F.avg(F.col("bot").cast("int")) * 100, 1).alias("bot_pct")
    )

sliding_query = sliding_df \
    .writeStream \
    .format("memory") \
    .queryName("sliding_30min_10min") \
    .outputMode("complete") \
    .start()

sliding_query.processAllAvailable()

spark.table("sliding_30min_10min") \
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "edits", "users", "pages", "bot_pct"
    ) \
    .orderBy("window_start") \
    .toPandas()

,window_start,window_end,edits,users,pages,bot_pct
0,2026-10-08 11:50:00,2026-10-08 12:20:00,773,399,612,4.7
1,2026-10-08 12:00:00,2026-10-08 12:30:00,1587,710,1171,6.1
2,2026-10-08 12:10:00,2026-10-08 12:40:00,2507,970,1852,6.2
3,2026-10-08 12:20:00,2026-10-08 12:50:00,2712,1057,2215,6.5
4,2026-10-08 12:30:00,2026-10-08 13:00:00,2015,778,1616,6.2
5,2026-10-08 12:40:00,2026-10-08 13:10:00,1095,469,919,6.0
6,2026-10-08 12:50:00,2026-10-08 13:20:00,117,87,119,8.5


In [11]:
tumbling_total = spark.table("tumbling_10min").agg(F.sum("edits")).first()[0]
sliding_total = spark.table("sliding_30min_10min").agg(F.sum("edits")).first()[0]
print(f"tumbling windows: {tumbling_total} edits counted")
print(f"sliding windows : {sliding_total} edits counted, {sliding_total / tumbling_total:.2f} times each edit")

tumbling windows: 3602 edits counted
sliding windows : 10806 edits counted, 3.00 times each edit


In [12]:
edit_size_df = events_df \
    .groupBy("wiki") \
    .agg(
        F.count("*").alias("edits"),
        F.sum(F.when(F.col("type") == "new", 1).otherwise(0)).alias("new_pages"),
        F.sum(F.when(F.col("size_delta") > 0, F.col("size_delta")).otherwise(0)).alias("bytes_added"),
        F.sum(F.when(F.col("size_delta") < 0, -F.col("size_delta")).otherwise(0)).alias("bytes_removed"),
        F.percentile_approx("size_delta", 0.5).alias("median_delta"),
        F.max("size_delta").alias("largest_addition"),
        F.min("size_delta").alias("largest_removal")
    )

edit_size_query = edit_size_df \
    .writeStream \
    .format("memory") \
    .queryName("edit_size_by_wiki") \
    .outputMode("complete") \
    .start()

edit_size_query.processAllAvailable()

spark.table("edit_size_by_wiki").orderBy(F.desc("edits")).toPandas()

,wiki,edits,new_pages,bytes_added,bytes_removed,median_delta,largest_addition,largest_removal
0,enwiki,2405,49,739954,207102,11,101552,-99166
1,frwiki,510,10,134320,105917,23,16363,-96836
2,dewiki,455,14,112707,29951,14,17647,-8252
3,eswiki,232,4,61294,29021,12,11126,-15282


In [13]:
top_pages_df = events_df \
    .groupBy("wiki", "title") \
    .agg(
        F.count("*").alias("edits"),
        F.approx_count_distinct("user").alias("users"),
        F.sum("size_delta").alias("net_bytes")
    )

top_pages_query = top_pages_df \
    .writeStream \
    .format("memory") \
    .queryName("top_pages") \
    .outputMode("complete") \
    .start()

top_pages_query.processAllAvailable()

spark.table("top_pages").orderBy(F.desc("edits"), "title").limit(10).toPandas()

,wiki,title,edits,users,net_bytes
0,enwiki,Linguistic history of India,17,1,3895
1,enwiki,Shinji Takashima (basketball),16,1,6274
2,enwiki,Atsuya Ogawa,15,1,8326
3,enwiki,Jaheru Yamauchi,14,1,10169
4,enwiki,List of Bhojpuri films,11,1,-175
5,enwiki,Mohamed Nasheed,10,2,118
6,enwiki,2027 AFC U-17 Women's Asian Cup qualification,9,3,823
7,enwiki,Environmental impacts of prehistoric humans,9,1,368
8,enwiki,Patrick Ferguson,9,1,244
9,frwiki,Brice Tarvel,8,1,118


In [14]:
followed_pages = [row["title"] for row in spark.table("top_pages").orderBy(F.desc("edits"), "title").limit(3).collect()]

followed_df = events_df \
    .filter(F.col("title").isin(followed_pages)) \
    .select("event_time", "wiki", "title", "user", "bot", "size_delta", "comment")

followed_query = followed_df \
    .writeStream \
    .format("memory") \
    .queryName("followed_pages") \
    .outputMode("append") \
    .start()

followed_query.processAllAvailable()

print(followed_pages)
spark.table("followed_pages").orderBy("event_time").toPandas()

['Linguistic history of India', 'Shinji Takashima (basketball)', 'Atsuya Ogawa']


,event_time,wiki,title,user,bot,size_delta,comment
0,2026-10-08 12:10:54,enwiki,Linguistic history of India,Bookaddictt,False,267,added citation
1,2026-10-08 12:11:21,enwiki,Linguistic history of India,Bookaddictt,False,192,added citation
2,2026-10-08 12:12:02,enwiki,Linguistic history of India,Bookaddictt,False,295,added citation
3,2026-10-08 12:12:43,enwiki,Linguistic history of India,Bookaddictt,False,247,added citation
4,2026-10-08 12:13:16,enwiki,Linguistic history of India,Bookaddictt,False,219,added citation
5,2026-10-08 12:13:35,enwiki,Shinji Takashima (basketball),ᱤᱧ ᱢᱟᱛᱟᱞ,False,5337,new
6,2026-10-08 12:14:14,enwiki,Shinji Takashima (basketball),ᱤᱧ ᱢᱟᱛᱟᱞ,False,356,ref
7,2026-10-08 12:14:37,enwiki,Shinji Takashima (basketball),ᱤᱧ ᱢᱟᱛᱟᱞ,False,35,"Set {{DEFAULTSORT}} to Takashima, Shinji usin..."
8,2026-10-08 12:14:45,enwiki,Linguistic history of India,Bookaddictt,False,-52,added sufficient inline citations
9,2026-10-08 12:15:04,enwiki,Shinji Takashima (basketball),ᱤᱧ ᱢᱟᱛᱟᱞ,False,26,added [[Category:2000 births]] using [[WP:HC|H...


In [15]:
for query in [tumbling_query, sliding_query, edit_size_query, top_pages_query, followed_query]:
    query.stop()